# Pregnancy Analysis with OMOPy

This notebook demonstrates `omopy.pregnancy` — identifying pregnancy episodes from OMOP CDM data using the HIPPS algorithm (Hierarchical Identification of Pregnancy Periods and States), summarising findings, and producing tables and plots.

**Covered:**
- Pregnancy outcome categories (`OUTCOME_CATEGORIES`)
- Identifying pregnancies with `identify_pregnancies` (HIPPS algorithm)
- Examining the `PregnancyResult` object
- Summarising with `summarise_pregnancies`
- Table output with `table_pregnancies`
- Plotting with `plot_pregnancies`
- Validating episodes with `validate_episodes`
- Mock data via `mock_pregnancy_cdm` for richer demonstrations

**Database:** Synthea DuckDB with 27 persons. Synthea data typically has minimal pregnancy records, so mock data is used for full demonstrations.

## 1. Configuration

In [1]:
DB_PATH = "../data/synthea.duckdb"
CDM_SCHEMA = "base"
WRITE_SCHEMA = "main"

## 2. Setup

Connect to the CDM and import the pregnancy module.

In [2]:
import datetime

from omopy.connector import cdm_from_con
from omopy.pregnancy import (
    OUTCOME_CATEGORIES,
    identify_pregnancies,
    mock_pregnancy_cdm,
    plot_pregnancies,
    summarise_pregnancies,
    table_pregnancies,
    validate_episodes,
)

print("Imports OK")

Imports OK


In [3]:
cdm = cdm_from_con(DB_PATH, cdm_schema=CDM_SCHEMA, write_schema=WRITE_SCHEMA)
print(cdm)
print(f"Persons: {cdm['person'].count()}")

CdmReference(name='dbt-synthea', version=5.4, source=duckdb, tables=36)
Persons: 27


## 3. Understanding the HIPPS Algorithm

The HIPPS algorithm (Smith et al. 2024) combines two complementary approaches to identify pregnancy episodes:

| Stage | Name | Description |
|-------|------|-------------|
| **HIP** | Outcome-anchored | Locates pregnancy outcome codes (live birth, stillbirth, etc.) and works backwards to estimate start date |
| **PPS** | Gestational-timing | Locates gestational age markers and estimates start from the timing |
| **Merge** | HIPPS merge | Combines HIP and PPS episodes, resolving conflicts |
| **ESD** | Episode Start Date | Refines start dates using supporting evidence (LMP records, prenatal visits) |

The `identify_pregnancies` function runs this full pipeline and returns a `PregnancyResult`.

## 4. Outcome Categories

`OUTCOME_CATEGORIES` lists the pregnancy outcome categories recognised by the algorithm.

In [4]:
print(f"Number of outcome categories: {len(OUTCOME_CATEGORIES)}")
print()
if isinstance(OUTCOME_CATEGORIES, dict):
    for code, label in OUTCOME_CATEGORIES.items():
        print(f"  {code:8s} -> {label}")
else:
    for i, cat in enumerate(OUTCOME_CATEGORIES, 1):
        print(f"  {i}. {cat}")

Number of outcome categories: 7

  LB       -> Live birth
  SB       -> Stillbirth
  AB       -> Abortion
  SA       -> Spontaneous abortion
  DELIV    -> Delivery (unspecified)
  ECT      -> Ectopic pregnancy
  PREG     -> Pregnancy (ongoing/unspecified)


## 5. Identifying Pregnancies (Synthea)

Run `identify_pregnancies` on the real Synthea data. With only 27 persons, Synthea may have few or no pregnancy records.

In [5]:
synthea_result = identify_pregnancies(cdm)
print(synthea_result)
print(f"Persons with pregnancies: {synthea_result.n_persons_input}")
print(f"Total episodes:          {synthea_result.n_episodes}")

episodes=shape: (0, 15)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ person_id ┆ merged_ep ┆ hip_episo ┆ pps_episo ┆ … ┆ esd_start ┆ gestation ┆ precision ┆ final_st │
│ ---       ┆ isode_id  ┆ de_id     ┆ de_id     ┆   ┆ _date     ┆ al_age_we ┆ ---       ┆ art_date │
│ i64       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ eks       ┆ str       ┆ ---      │
│           ┆ i64       ┆ i64       ┆ i64       ┆   ┆ date      ┆ ---       ┆           ┆ date     │
│           ┆           ┆           ┆           ┆   ┆           ┆ f64       ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
└───────────┴───────────┴───────────┴───────────┴───┴───────────┴───────────┴───────────┴──────────┘ hip_episodes=shape: (0, 7)
┌───────────┬────────────┬──────────┬───────────────┬───────────────┬──────────────┬───────────────┐
│ person_id ┆ episode_id ┆ category ┆ ep

In [6]:
# Show episodes if any were found
if synthea_result.n_episodes > 0:
    print(synthea_result.episodes)
else:
    print("No pregnancy episodes found in Synthea data.")
    print("This is expected — we will use mock data below.")

No pregnancy episodes found in Synthea data.
This is expected — we will use mock data below.


## 6. Using Mock Data

`mock_pregnancy_cdm` generates a synthetic CDM with pregnancy-related records — useful for demonstrating the full pipeline when real data is sparse.

In [7]:
mock_cdm = mock_pregnancy_cdm(seed=42, n_persons=100)
print(mock_cdm)

CdmReference(name='mock_pregnancy', version=5.4, source=local, tables=6)


In [8]:
mock_result = identify_pregnancies(mock_cdm)
print(mock_result)
print(f"Persons with pregnancies: {mock_result.n_persons_input}")
print(f"Total episodes:          {mock_result.n_episodes}")

episodes=shape: (106, 15)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ person_id ┆ merged_ep ┆ hip_episo ┆ pps_episo ┆ … ┆ esd_start ┆ gestation ┆ precision ┆ final_st │
│ ---       ┆ isode_id  ┆ de_id     ┆ de_id     ┆   ┆ _date     ┆ al_age_we ┆ ---       ┆ art_date │
│ i64       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ eks       ┆ str       ┆ ---      │
│           ┆ i64       ┆ i64       ┆ i64       ┆   ┆ date      ┆ ---       ┆           ┆ date     │
│           ┆           ┆           ┆           ┆   ┆           ┆ f64       ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 2         ┆ 1         ┆ 1         ┆ 1         ┆ … ┆ null      ┆ null      ┆ low       ┆ 2015-10- │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ 03       │
│ 3         ┆ 2         ┆ 2         ┆ 2         ┆ … ┆ 2015-05-2 ┆

## 7. Examining PregnancyResult

`PregnancyResult` holds episode DataFrames (Polars), pipeline settings, and summary counts.

In [9]:
# Summary counts
print(f"Person count:  {mock_result.n_persons_input}")
print(f"Episode count: {mock_result.n_episodes}")

Person count:  72
Episode count: 106


In [10]:
# Pipeline settings used for identification
print("Settings:")
print(mock_result.settings)

Settings:
{'start_date': None, 'end_date': None, 'age_bounds': [10, 55], 'just_gestation': True, 'min_cell_count': 5}


In [11]:
# The .episodes attribute contains the final episodes DataFrame
print(f"Episodes shape: {mock_result.episodes.shape}")
print(f"Columns: {mock_result.episodes.columns}")
print()
print(mock_result.episodes.head(10))

Episodes shape: (106, 15)
Columns: ['person_id', 'merged_episode_id', 'hip_episode_id', 'pps_episode_id', 'category', 'episode_start_date', 'episode_end_date', 'outcome_date', 'outcome_concept_id', 'n_pps_records', 'source', 'esd_start_date', 'gestational_age_weeks', 'precision', 'final_start_date']

shape: (10, 15)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ person_id ┆ merged_ep ┆ hip_episo ┆ pps_episo ┆ … ┆ esd_start ┆ gestation ┆ precision ┆ final_st │
│ ---       ┆ isode_id  ┆ de_id     ┆ de_id     ┆   ┆ _date     ┆ al_age_we ┆ ---       ┆ art_date │
│ i64       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ eks       ┆ str       ┆ ---      │
│           ┆ i64       ┆ i64       ┆ i64       ┆   ┆ date      ┆ ---       ┆           ┆ date     │
│           ┆           ┆           ┆           ┆   ┆           ┆ f64       ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════

In [12]:
# The .merged_episodes attribute contains pre-ESD merged HIP+PPS episodes
print(f"Merged episodes shape: {mock_result.merged_episodes.shape}")
print(f"Columns: {mock_result.merged_episodes.columns}")
print()
print(mock_result.merged_episodes.head(10))

Merged episodes shape: (106, 11)
Columns: ['person_id', 'merged_episode_id', 'hip_episode_id', 'pps_episode_id', 'category', 'episode_start_date', 'episode_end_date', 'outcome_date', 'outcome_concept_id', 'n_pps_records', 'source']

shape: (10, 11)
┌───────────┬────────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬─────────┐
│ person_id ┆ merged_epi ┆ hip_episo ┆ pps_episo ┆ … ┆ outcome_d ┆ outcome_c ┆ n_pps_rec ┆ source  │
│ ---       ┆ sode_id    ┆ de_id     ┆ de_id     ┆   ┆ ate       ┆ oncept_id ┆ ords      ┆ ---     │
│ i64       ┆ ---        ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ str     │
│           ┆ i64        ┆ i64       ┆ i64       ┆   ┆ date      ┆ i64       ┆ i64       ┆         │
╞═══════════╪════════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═════════╡
│ 2         ┆ 1          ┆ 1         ┆ 1         ┆ … ┆ 2016-08-0 ┆ 4142115   ┆ 3         ┆ HIP+PPS │
│           ┆            ┆           ┆      

## 8. Summarise Pregnancies

`summarise_pregnancies` converts the `PregnancyResult` into the standard `SummarisedResult` format, with counts, proportions, and gestational age statistics.

In [13]:
summary = summarise_pregnancies(mock_result)
print(type(summary))
print(f"Rows: {summary.data.height}")

<class 'omopy.generics.summarised_result.SummarisedResult'>
Rows: 30


In [14]:
# View summarised data
print(summary.data.head(15))

shape: (15, 13)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ result_id ┆ cdm_name  ┆ group_nam ┆ group_lev ┆ … ┆ variable_ ┆ estimate_ ┆ estimate_ ┆ estimate │
│ ---       ┆ ---       ┆ e         ┆ el        ┆   ┆ level     ┆ name      ┆ type      ┆ _value   │
│ i64       ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 1         ┆ mock_preg ┆ overall   ┆ overall   ┆ … ┆           ┆ count     ┆ integer   ┆ 106      │
│           ┆ nancy     ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ 1         ┆ mock_preg ┆ overall   ┆ overall   ┆ … ┆           ┆ count     ┆ integer   ┆ 72       │
│           ┆ nancy     ┆           ┆           ┆   ┆           ┆          

In [15]:
# Summarise with strata (e.g., by outcome category)
summary_stratified = summarise_pregnancies(
    mock_result,
    strata=["outcome_category"],
)
print(f"Stratified rows: {summary_stratified.data.height}")
print(summary_stratified.data.head(15))

Stratified rows: 30
shape: (15, 13)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ result_id ┆ cdm_name  ┆ group_nam ┆ group_lev ┆ … ┆ variable_ ┆ estimate_ ┆ estimate_ ┆ estimate │
│ ---       ┆ ---       ┆ e         ┆ el        ┆   ┆ level     ┆ name      ┆ type      ┆ _value   │
│ i64       ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 1         ┆ mock_preg ┆ overall   ┆ overall   ┆ … ┆           ┆ count     ┆ integer   ┆ 106      │
│           ┆ nancy     ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ 1         ┆ mock_preg ┆ overall   ┆ overall   ┆ … ┆           ┆ count     ┆ integer   ┆ 72       │
│           ┆ nancy     ┆           ┆           ┆   ┆  

In [16]:
# Summarise with default settings (no min_cell_count parameter needed)
summary_default = summarise_pregnancies(mock_result)
print(f"Default summary rows: {summary_default.data.height}")
print(summary_default.data.head(10))

Default summary rows: 30
shape: (10, 13)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ result_id ┆ cdm_name  ┆ group_nam ┆ group_lev ┆ … ┆ variable_ ┆ estimate_ ┆ estimate_ ┆ estimate │
│ ---       ┆ ---       ┆ e         ┆ el        ┆   ┆ level     ┆ name      ┆ type      ┆ _value   │
│ i64       ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 1         ┆ mock_preg ┆ overall   ┆ overall   ┆ … ┆           ┆ count     ┆ integer   ┆ 106      │
│           ┆ nancy     ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
│ 1         ┆ mock_preg ┆ overall   ┆ overall   ┆ … ┆           ┆ count     ┆ integer   ┆ 72       │
│           ┆ nancy     ┆           ┆           ┆ 

## 9. Pregnancy Table

`table_pregnancies` formats the summarised result as a publication-ready table. Use `type="gt"` for a `great_tables` GT object or `type="polars"` for a plain DataFrame.

In [17]:
# GT table (renders as rich HTML in notebooks)
tbl_gt = table_pregnancies(summary, type="gt")
tbl_gt

cdm_name,variable_name,variable_level,estimate_name,estimate_value
mock_pregnancy,Number episodes,,count,106
mock_pregnancy,Number persons,,count,72
mock_pregnancy,Outcome category,AB,count,7
mock_pregnancy,Outcome category,AB,percentage,6.6
mock_pregnancy,Outcome category,DELIV,count,57
mock_pregnancy,Outcome category,DELIV,percentage,53.8
mock_pregnancy,Outcome category,ECT,count,8
mock_pregnancy,Outcome category,ECT,percentage,7.5
mock_pregnancy,Outcome category,PREG,count,14
mock_pregnancy,Outcome category,PREG,percentage,13.2


In [18]:
# Stratified table
tbl_stratified = table_pregnancies(summary_stratified, type="gt")
tbl_stratified

cdm_name,variable_name,variable_level,estimate_name,estimate_value
mock_pregnancy,Number episodes,,count,106
mock_pregnancy,Number persons,,count,72
mock_pregnancy,Outcome category,AB,count,7
mock_pregnancy,Outcome category,AB,percentage,6.6
mock_pregnancy,Outcome category,DELIV,count,57
mock_pregnancy,Outcome category,DELIV,percentage,53.8
mock_pregnancy,Outcome category,ECT,count,8
mock_pregnancy,Outcome category,ECT,percentage,7.5
mock_pregnancy,Outcome category,PREG,count,14
mock_pregnancy,Outcome category,PREG,percentage,13.2


## 10. Pregnancy Plot

`plot_pregnancies` creates interactive Plotly figures for visualising pregnancy outcomes, gestational age, and timelines.

In [19]:
fig = plot_pregnancies(summary)
fig.show()

In [20]:
# Plot the stratified summary
fig_strat = plot_pregnancies(summary_stratified)
fig_strat.show()

## 11. Validating Episodes

`validate_episodes` checks pregnancy episodes for consistency issues such as gestational periods exceeding expected bounds or overlapping episodes for the same person.

In [21]:
# Validate episode temporal consistency
issues = validate_episodes(mock_result.episodes)
print(type(issues))
print(issues)

<class 'polars.dataframe.frame.DataFrame'>
shape: (3, 3)
┌──────────────────┬──────────────┬───────────────────────────────┐
│ check            ┆ n_violations ┆ details                       │
│ ---              ┆ ---          ┆ ---                           │
│ str              ┆ i64          ┆ str                           │
╞══════════════════╪══════════════╪═══════════════════════════════╡
│ start_before_end ┆ 0            ┆ 0 episodes with start > end   │
│ max_duration     ┆ 0            ┆ 0 episodes exceeding 365 days │
│ no_overlaps      ┆ 9            ┆ 9 overlapping episode pairs   │
└──────────────────┴──────────────┴───────────────────────────────┘


## 12. Advanced Options

`identify_pregnancies` supports several parameters to refine the identification:

| Parameter | Default | Description |
|-----------|---------|-------------|
| `start_date` | `None` | Study window start (`datetime.date` or `None` for all) |
| `end_date` | `None` | Study window end (`datetime.date` or `None` for all) |
| `age_bounds` | `(10, 55)` | Min/max age at pregnancy start |
| `just_gestation` | `True` | If `True`, only use gestation-related evidence |

In [22]:
# Restrict to a date window
result_window = identify_pregnancies(
    mock_cdm,
    start_date=datetime.date(2018, 1, 1),
    end_date=datetime.date(2022, 12, 31),
)
print(f"Episodes in 2018-2022: {result_window.n_episodes}")
print(f"Persons:               {result_window.n_persons_input}")

Episodes in 2018-2022: 43
Persons:               72


In [23]:
# Narrow the age range
result_age = identify_pregnancies(
    mock_cdm,
    age_bounds=(18, 40),
)
print(f"Episodes (age 18-40): {result_age.n_episodes}")
print(f"Persons:              {result_age.n_persons_input}")

Episodes (age 18-40): 106
Persons:              72


In [24]:
# Include non-gestation evidence
result_full = identify_pregnancies(
    mock_cdm,
    just_gestation=False,
)
print(f"Episodes (just_gestation=True):  {mock_result.n_episodes}")
print(f"Episodes (just_gestation=False): {result_full.n_episodes}")

Episodes (just_gestation=True):  106
Episodes (just_gestation=False): 106


In [25]:
# Combine all options
result_combined = identify_pregnancies(
    mock_cdm,
    start_date=datetime.date(2015, 1, 1),
    end_date=datetime.date(2023, 12, 31),
    age_bounds=(15, 45),
    just_gestation=False,
)
print(f"Combined options — episodes: {result_combined.n_episodes}")
print(f"Combined options — persons:  {result_combined.n_persons_input}")

Combined options — episodes: 106
Combined options — persons:  72


## 13. Summary

This notebook covered the full `omopy.pregnancy` workflow:

1. **`OUTCOME_CATEGORIES`** — pregnancy outcome categories (LB, SB, AB, SA, DELIV, ECT, PREG).
2. **`identify_pregnancies`** — runs the HIPPS algorithm to extract pregnancy episodes, returning a `PregnancyResult` with `.episodes`, `.hip_episodes`, `.pps_episodes`, `.merged_episodes`, `.settings`, `.n_persons_input`, and `.n_episodes`.
3. **`summarise_pregnancies`** — converts results into the standard `SummarisedResult` format, with optional strata and cell-count suppression.
4. **`table_pregnancies`** — formats results as GT or polars tables.
5. **`plot_pregnancies`** — interactive Plotly visualisations.
6. **`validate_episodes`** — checks for consistency issues in identified episodes.
7. **`mock_pregnancy_cdm`** — generates synthetic CDM data for testing and demonstrations.

**Next steps:**
- Combine with `omopy.cohort` to build pregnancy-based cohorts
- Use `omopy.characteristics` to characterise pregnancy cohorts
- Export `SummarisedResult` to CSV/parquet for downstream reporting